In [ ]:
import os
import pickle
import random
import sys
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# SNOMED CT Methods - Expert Examples

This notebook demonstrates advanced SNOMED CT methods for clinical concept expansion and similarity analysis using the `snomed_methods_v1` package.

## Setup

Ensure the package is on the Python path.

In [ ]:
# Add project root to path if not already present
project_root = "/workspaces/snomed_methods"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.snomed_methods.snomed_methods_v1 import SnomedRelations

## Configuration

Set the paths to your SNOMED CT data. By default, we use the included UK Clinical RF2 dataset.

### Path Variables
- `SNOMED_DIR`: Base directory for SNOMED CT data (default: `/workspaces/snomed_methods/uk_sct2cl_42.2.0`)
- `SCT2_PATH`: Stated relationships file path (auto-computed from SNOMED_DIR)
- `UK_SNOMED_DIR`: UK Clinical RF2 directory (auto-computed)

In [ ]:
# Verify paths exist
path_status = []
for _path_name, path_value in [
    ("SNOMED Directory", DEFAULT_SNOMED_DIR),
    ("Stated Relationships", DEFAULT_SCT2_PATH),
    ("UK SNOMED Directory", DEFAULT_UK_SNOMED_DIR),
]:
    exists = "exists" if os.path.exists(path_value) else "NOT FOUND"
    path_status.append((_path_name, path_value, exists))

# Print status
for name, path, status in path_status:
    print(f"{name}: {status}")

## Initialize SnomedRelations Object

The `SnomedRelations` class provides methods for exploring SNOMED CT concept relationships.

**Important Notes:**
- Without MedCAT, concept name retrieval will return `None`
- The relationship data files are large (hundreds of MB). Ensure your system has sufficient memory.

In [ ]:
# Initialize SnomedRelations with UK Clinical RF2 data
# Note: medcat=True requires a MedCAT model to retrieve concept names
snomed_relations_obj = SnomedRelations(
    snomed_rf2_full_path=DEFAULT_SCT2_PATH, dhcap02=True
)  # Default configuration

## Example 1: Recursive Code Expansion

Find related SNOMED CT codes by expanding from a starting concept using parent-child relationships.

This method traverses the SNOMED CT hierarchy in both directions (parents and children).

In [ ]:
# Define a starting SNOMED concept code (CUI)
# 399187006 = "Hemochromatosis"
outcome_variable_cui_for_filter = "399187006"

In [ ]:
# Perform recursive expansion
# n_recursion determines how many degrees of separation to explore
filter_root_cui = outcome_variable_cui_for_filter

retrieved_codes_snomed_tree, retrieved_names_snomed_tree = (
    snomed_relations_obj.recursive_code_expansion(
        filter_root_cui,
        n_recursion=3,  # Search up to 10 degrees of separation
        debug=False,
    )
)


# Show sample results
if retrieved_codes_snomed_tree:
    for _i, (_code, _name) in enumerate(
        list(zip(retrieved_codes_snomed_tree, retrieved_names_snomed_tree))[:5], 1
    ):
        pass

### Understanding Recursion Depth

The `n_recursion` parameter controls how many generations of parent/child relationships to explore:

| Level | Description |
|-------|-------------|
| 1-3   | Direct parent/child relationships only |
| 4-7   | Extended family within a few generations |
| 8+    | Broad exploration, may include loosely related concepts |

**Recommendation:** Start with `n_recursion=5` and increase if you need more comprehensive coverage.

## Example 2: MedCAT Concept Similarity

Use MedCAT's context vectors to find conceptually similar terms. This requires a trained MedCAT model with context embeddings.

**Note:** This example is included for reference. Without a MedCAT model, this method will fail gracefully.

In [ ]:
# Try MedCAT-based similarity (requires MedCAT model)
try:
    retrieved_codes_medcat_cdb, retrieved_names_medcat_cdb = (
        snomed_relations_obj.get_medcat_cdb_most_similar(
            filter_root_cui,
            context_type="xxxlong",  # Long context window for better coverage
            type_id_filter=[],  # No type filtering - consider all concepts
            topn=50,  # Return 50 most similar concepts
        )
    )

    if retrieved_codes_medcat_cdb:
        for _i, (_code, _name) in enumerate(
            list(zip(retrieved_codes_medcat_cdb, retrieved_names_medcat_cdb))[:5], 1
        ):
            pass

except Exception:
    pass

## Example 3: Batch Processing

Process multiple starting concepts and export results to CSV for later analysis.

This pattern is useful when you need to:
- Process a predefined list of target concepts
- Generate concept expansions for multiple conditions
- Pipeline results into downstream analysis

In [ ]:
# Define a batch of SNOMED concept codes
ronnie_code_list = [
    700065003,  # Iron deficiency anemia
    471885006,  # Chronic viral hepatitis
    890122001,  # Primary malignant neoplasm
    890119003,  # Benign neoplasm
    871638006,  # Inflammatory disease
    890121008,  # Metastatic malignant neoplasm
    871649000,  # Primary benign neoplasm
    890120009,  # Secondary malignant neoplasm
    472316006,  # Autoimmune disease
    45227007,  # Genital neoplasm
]

### Batch Processing Function

Helper function to process multiple concepts and build a results DataFrame.

In [ ]:
def create_dataframe_snomed(
    input_codes: List[int], n_recursion: int = 20
) -> pd.DataFrame:
    """
    Process multiple starting codes and build a results DataFrame.

    Args:
        input_codes: List of SNOMED CT concept codes (integers)
        n_recursion: Recursion depth for expansion

    Returns:
        DataFrame with one row per starting code containing:
        - filter_root_cui: The starting concept code
        - retrieved_codes_count: Number of related concepts found
        - retrieved_names_count: Number of concept names retrieved
    """
    records = []

    for _i, filter_root_cui in enumerate(input_codes, 1):

        try:
            retrieved_codes, retrieved_names = (
                snomed_relations_obj.recursive_code_expansion(
                    filter_root_cui, n_recursion=n_recursion, debug=False
                )
            )

            record = {
                "filter_root_cui": str(filter_root_cui),
                "retrieved_codes_count": len(retrieved_codes) if retrieved_codes else 0,
                "retrieved_names_count": len(retrieved_names) if retrieved_names else 0,
            }
            records.append(record)

        except Exception:
            pass

    return pd.DataFrame(records)

### Run Batch Processing

Process all concepts and save results to CSV.

In [ ]:
# Process the batch
result_df = create_dataframe_snomed(ronnie_code_list, n_recursion=5)


# Save results
output_path = "/workspaces/snomed_methods/result_snomed_batch.csv"
result_df.to_csv(output_path, index=False)

# Display summary
display(Markdown("### Batch Processing Results Summary"))

## Example 4: Embedding-Based Similarity Analysis

Use Gatortron embeddings (clinical text transformer model) to find SNOMED concepts with similar clinical meanings.

This method:
- Uses pretrained clinical language models
- Captures semantic similarity beyond hierarchical relationships
- Can identify conceptually related but structurally distant concepts

### Requirement
Pre-computed embedding file: `gatortron_embeddings.pkl` containing a dictionary mapping terms to embedding vectors.

In [ ]:
# Path to pre-computed Gatortron embeddings
GATORTRON_EMBEDDING_PATH = os.environ.get(
    "GATORTRON_EMBEDDING_PATH",
    "/workspaces/snomed_methods/uk_sct2cl_42.2.0/gatortron_embeddings.pkl",
)

# Check if embedding file exists
if os.path.exists(GATORTRON_EMBEDDING_PATH):

    with open(GATORTRON_EMBEDDING_PATH, "rb") as f:
        loaded_dict = pickle.load(f)

    # Show sample keys
else:
    pass

### Cosine Similarity Function

Find the most similar concepts to a target concept using cosine similarity on embedding vectors.

In [ ]:
def find_most_similar(
    target_vector: np.ndarray, term_vectors: Dict[str, np.ndarray], n: int = 5
) -> List[Tuple[str, float]]:
    """
    Find the n most similar vectors to the target_vector using cosine similarity.

    Args:
        target_vector: The reference embedding vector (1D numpy array)
        term_vectors: Dictionary mapping terms to their embeddings (term -> ndarray)
        n: Number of similar terms to return

    Returns:
        List of (term, similarity_score) tuples sorted by descending similarity
    """
    # Reshape for sklearn cosine_similarity
    target_vector = target_vector.reshape(1, -1)

    similarities = {}

    for term, vector in tqdm(
        term_vectors.items(), desc="Calculating similarities", leave=False
    ):
        vector = vector.reshape(1, -1)
        similarity_score = cosine_similarity(target_vector, vector)[0, 0]
        similarities[term] = similarity_score

    # Sort by similarity (descending order)
    sorted_terms = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

    return sorted_terms[:n]


# Example usage
if os.path.exists(GATORTRON_EMBEDDING_PATH):
    target_term = "hemochromatosis"

    if target_term in loaded_dict:
        target_vector = loaded_dict[target_term]

        # Get top 50 similar terms (using a subset for speed)
        all_keys = list(loaded_dict.keys())

        # Use random sample for demonstration (full dataset would be ~1 hour)

        selected_keys = random.sample(all_keys, sample_size)

        filtered_dict = {k: loaded_dict[k] for k in selected_keys}

    else:
        pass

## Example 5: Comprehensive Concept Analysis

Combine multiple methods for comprehensive concept analysis.

This approach:
1. Expands the concept tree to find related concepts
2. Uses embeddings to find semantically similar concepts
3. Provides a unified analysis output

In [ ]:
def analyze_concept_comprehensive(
    cui: str, n_tree_recursion: int = 10, sample_size: int = 5000
) -> Dict:
    """
    Perform comprehensive analysis on a SNOMED CT concept using multiple methods.

    Args:
        cui: Starting concept code (string or integer)
        n_tree_recursion: Recursion depth for tree expansion
        sample_size: Sample size for embedding similarity (for performance)

    Returns:
        Dictionary containing results from all methods with keys:
        - cui: Original concept code
        - tree_expansion: Dict with codes, names, and count from recursive search
        - embedding_similarity: List of (term, score) tuples
    """
    results = {
        "cui": str(cui),
        "tree_expansion": {"codes": [], "names": [], "count": 0},
        "embedding_similarity": [],
    }

    # Method 1: Tree expansion via parent-child relationships
    try:
        codes, names = snomed_relations_obj.recursive_code_expansion(
            str(cui), n_recursion=n_tree_recursion, debug=False
        )
        results["tree_expansion"] = {
            "codes": list(codes) if codes else [],
            "names": list(names) if names else [],
            "count": len(codes) if codes else 0,
        }
    except Exception:
        pass

    # Method 2: Embedding similarity (if embeddings available)
    if os.path.exists(GATORTRON_EMBEDDING_PATH):
        try:
            all_keys = list(loaded_dict.keys())
            sample_size = min(sample_size, len(all_keys))
            selected_keys = random.sample(all_keys, sample_size)

            filtered_dict = {k: loaded_dict[k] for k in selected_keys}

            # Use a representative clinical term as proxy
            if "hemochromatosis" in loaded_dict:
                target_vector = loaded_dict["hemochromatosis"]
                result = find_most_similar(target_vector, filtered_dict, n=50)
                results["embedding_similarity"] = result[:10]
        except Exception:
            pass

    return results

### Run Comprehensive Analysis

Perform combined tree expansion and embedding similarity analysis.

In [ ]:
# Run comprehensive analysis on a test concept
test_cui = "399187006"  # Hemochromatosis


try:
    analysis_results = analyze_concept_comprehensive(
        test_cui,
        n_tree_recursion=5,  # Lower recursion for faster demo
        sample_size=1000,  # Smaller sample size for speed
    )

    if analysis_results["embedding_similarity"]:

        for _i, (_term, _score) in enumerate(
            analysis_results["embedding_similarity"][:5], 1
        ):
            pass

except Exception:
    pass

## Quick Reference

### Environment Variables

Set these before running the notebook to customize paths:

| Variable | Default | Description |
|----------|---------|-------------|
| `SNOMED_DIR` | `/workspaces/snomed_methods/uk_sct2cl_42.2.0` | Base directory for SNOMED CT data |
| `SCT2_PATH` | Auto-computed from SNOMED_DIR | Path to sct2_StatedRelationship_Full_INT_*.txt |
| `UK_SNOMED_DIR` | Auto-computed | Path to UK Clinical RF2 directory |
| `MEDCAT_MODEL_PATH` | None | Path to MedCAT model pack (optional) |
| `GATORTRON_EMBEDDING_PATH` | `/workspaces/snomed_methods/uk_sct2cl_42.2.0/gatortron_embeddings.pkl` | Path to pre-computed embeddings |

### Key Methods

| Method | Parameters | Description |
|--------|------------|-------------|
| `recursive_code_expansion(cui, n_recursion)` | cui: str, n_recursion: int | Find related concepts via parent-child relationships |
| `get_medcat_cdb_most_similar(cui, topn)` | cui: str, context_type: str, type_id_filter: list, topn: int | Find conceptually similar terms using MedCAT embeddings |

### Output Files

| File | Description |
|------|-------------|
| `result_snomed_batch.csv` | Batch processing results (codes found per starting concept) |

## Next Steps

For more information and examples:

1. **Basic Term Lookup**: See `notebooks/term_lookup_examples.ipynb`
2. **Detailed Examples**: Check `examples/example_term_lookup.py`
3. **API Documentation**: Read `TERM_LOOKUP_README.md`

### Generating Gatortron Embeddings

To generate embeddings for your SNOMED concepts:

```python
from snomed_term_lookup import create_term_lookup_from_directory

# Load SNOMED data
lookup = create_term_lookup_from_directory("/path/to/snomed/data")

# Generate term list
all_terms = []
for cui in lookup.concept_df["conceptId"].unique():
    names = lookup.getconcept_info(str(cui))["all_names"]
    all_terms.extend(names)

# Process with Gatortron model to generate embeddings...
```

In [ ]:
# Verification: Check output files exist
import os

output_files = ["/workspaces/snomed_methods/result_snomed_batch.csv"]

for filepath in output_files:
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"{filepath}: {size} bytes")
    else:
        print(f"{filepath}: NOT FOUND")